# 🥇 Gold Layer — Business Aggregations

> **Purpose**: Produce Power BI-ready fact and dimension tables in a star schema.
> All tables are optimised with ZORDER for DirectLake performance.

## Tables Created
| Table | Type | Description |
|---|---|---|
| `gold.fact_sales` | Fact | Enriched sales transactions |
| `gold.agg_sales_monthly` | Aggregate | Pre-computed monthly KPIs |
| `gold.dim_date` | Dimension | Calendar dimension (2018–2030) |

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────
dbutils.widgets.text('storage_account','<storage_account>','ADLS Gen2 Account Name')
dbutils.widgets.text('batch_id',       '',                 'Batch ID')

storage_account = dbutils.widgets.get('storage_account')
batch_id        = dbutils.widgets.get('batch_id') or spark.sparkContext.applicationId

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net'
gold_path   = f'abfss://gold@{storage_account}.dfs.core.windows.net'

spark.conf.set('spark.sql.shuffle.partitions', '8')
print('Gold layer initialised')

In [ ]:
# ── Build fact_sales ───────────────────────────────────────────────────────
df_silver_sales = (
    spark.read.format('delta').load(f'{silver_path}/dbo_sales')
    .filter(F.col('_is_current') == True)
)

# Enrich with dimension lookups (join keys only — no fan-out)
df_fact = (
    df_silver_sales
    .select(
        'sale_id', 'customer_id', 'product_id', 'sale_date',
        'quantity', 'unit_price', 'revenue', 'region'
    )
    # Derive date key for star schema join
    .withColumn('date_key', F.date_format(F.col('sale_date'), 'yyyyMMdd').cast('integer'))
    .withColumn('year_month', F.date_format(F.col('sale_date'), 'yyyy-MM'))
    .withColumn('_gold_loaded_at', F.current_timestamp())
    .withColumn('_batch_id', F.lit(batch_id))
)

(
    df_fact.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .option('path', f'{gold_path}/fact_sales')
    .saveAsTable('gold.fact_sales')
)

spark.sql('OPTIMIZE gold.fact_sales ZORDER BY (sale_date, region)')
print(f'fact_sales: {df_fact.count():,} rows')

In [ ]:
# ── Build agg_sales_monthly ────────────────────────────────────────────────
df_agg = (
    df_fact
    .groupBy('year_month', 'region')
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.sum('quantity').alias('total_units'),
        F.countDistinct('sale_id').alias('order_count'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.avg('revenue').alias('avg_order_value'),
        F.max('revenue').alias('max_order_value'),
        F.min('sale_date').alias('period_start'),
        F.max('sale_date').alias('period_end')
    )
    .withColumn('total_revenue',   F.round(F.col('total_revenue'), 2))
    .withColumn('avg_order_value', F.round(F.col('avg_order_value'), 2))
    .withColumn('_gold_loaded_at', F.current_timestamp())
)

(
    df_agg.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .option('path', f'{gold_path}/agg_sales_monthly')
    .saveAsTable('gold.agg_sales_monthly')
)

spark.sql('OPTIMIZE gold.agg_sales_monthly ZORDER BY (year_month, region)')
print(f'agg_sales_monthly: {df_agg.count():,} rows')

In [ ]:
# ── Build dim_date ─────────────────────────────────────────────────────────
# Generate a complete date spine from 2018 to 2030
from pyspark.sql.types import DateType
import datetime

start_date = datetime.date(2018, 1, 1)
end_date   = datetime.date(2030, 12, 31)
date_count = (end_date - start_date).days + 1

df_dates = (
    spark.range(date_count)
    .withColumn('date', (F.lit(start_date.strftime('%Y-%m-%d')).cast(DateType()) + F.col('id').cast('int')))
    .withColumn('date_key',      F.date_format('date', 'yyyyMMdd').cast('integer'))
    .withColumn('year',          F.year('date'))
    .withColumn('quarter',       F.quarter('date'))
    .withColumn('month_num',     F.month('date'))
    .withColumn('month_name',    F.date_format('date', 'MMMM'))
    .withColumn('month_short',   F.date_format('date', 'MMM'))
    .withColumn('year_month',    F.date_format('date', 'yyyy-MM'))
    .withColumn('week_of_year',  F.weekofyear('date'))
    .withColumn('day_of_week',   F.dayofweek('date'))
    .withColumn('day_name',      F.date_format('date', 'EEEE'))
    .withColumn('is_weekend',    (F.dayofweek('date').isin([1, 7])).cast('boolean'))
    .withColumn('is_month_end',  (F.col('date') == F.last_day('date')).cast('boolean'))
    .withColumn('fiscal_year',   F.when(F.month('date') >= 7, F.year('date') + 1).otherwise(F.year('date')))
    .drop('id')
)

(
    df_dates.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .option('path', f'{gold_path}/dim_date')
    .saveAsTable('gold.dim_date')
)

spark.sql('OPTIMIZE gold.dim_date ZORDER BY (date_key)')
print(f'dim_date: {df_dates.count():,} rows (2018–2030)')
print('\nGold layer complete — all tables ready for Power BI DirectLake')